In [13]:

import requests as req
import bs4
from bs4 import BeautifulSoup
import sys
import os
import time
import random
import xml.etree.ElementTree as ET
import lxml.etree
from lxml import etree

url_web = ''
img_url,price,title,brand,category,page_url,description=[],[],[],[],[],[],[]

In [ ]:
from bs4 import BeautifulSoup
import bs4
def extract_page(url):      #function to extract page
    global url_web
    if 'jumia.co.ke/' in url:
        url_web = 'https://www.jumia.co.ke'
    else:
        print('This URL is not related to jumia website')
        sys.exit()
        
    resp = req.get(url)
    page = bs4.BeautifulSoup(resp.content, 'lxml')
    return page

In [15]:
def preprocess_data(page):          #function to preprocess/ extract information
    global url_web
    products = page.findall('div', attrs={'class': '-pax row_no-g_4cl_3cm-shs'})
    if len(products) > 0:
        k = 1
        try:
            for product in products[0].find_all('articles'):
                product_page = extract_page(
                    url_web + product.a['href'])  #crawls product url page
                
                desc = product_page.find('div', attrs = {'class': 'markup -mhm -pvl -oxa -sc'})
                
                print(k, 'Product Page Url: ', url_web + product.a['href'])
                print(' Product Image Url: ', product.img['data-src'])
                print(' Product Category: ', product.a['data-category'])
                print(' Product Name: ', product.a['data-name'])
                print(' Product Brand: ', product.a['data-brand'])
                
                if url_web == 'http://www.jumia.co.ke':
                    print(' Product Price: ', int(product.find('div', attrs={'class': 'prc'}).text[2:].strip().replace(',', '')))
                    price.append(int(product.find('div', attrs ={'class': 'prc'}).text[2:].strip().replace(',', '')))
                else:
                    print(' Product Price: ', int(product.find('div', attrs={'class': 'prc'}).text[3:].strip().replace(',', '')))
                    price.append(int(product.find('div', attrs={'class': 'prc'}).text[3:].strip().replace(',', '')))
                    
                print(' Product Description: ', desc.text)
                print('' * 2), print('\t\t**************************************************************\n')
                k += 1
                page_url.append(url_web + product.a['href'])         #image url
                img_url.append(product.img['data-src'])
                category.append(product.a['data-category'])
                title.append(product.a['data-name'])
                brand.append(product.a['data-brand'])
                description.append(desc.text)
                
                time.sleep(random.randint(5, 15))
        except:
            pass
    else:
        return -1

In [16]:
def scrape_web(url, loop_iter=-1):      #pagination function #dividing large sets of data into smaller, more managable pages
    if loop_iter == -1:            #if pagination by default
        p = 1
        while True:
            page = extract_page(url + '?page='+str(p) + '#catalog-listing')
            status = preprocess_data(page)
            if status == -1:
                break
            p += 1
    else:
        for i in range(1, loop_iter +1):
            page = extract_page(url + '?page='+str(i) + '#catalog-listing')
            status = preprocess_data(page)
            if status == -1:
                break

In [17]:
# create XML file
def create_xml(filepath):
    root = ET.Element('products')       #we make root element
    
    for user in range(len(price)):
        product = ET.SubElement(root, 'product')
        
        c1 = ET.SubElement(product, 'storefront_category')
        c1.text = str(category[user])
        b = ET.SubElement(product, 'storefront_brand_name')
        b.text = str(brand[user])
        n1 = ET.SubElement(product, 'product_name')
        n1.text = str(title[user])
        p_url = ET.SubElement(product, 'product_url')
        p_url.text = str(page_url[user])
        i_url = ET.SubElement(product, 'image_url')
        i_url.text = str(img_url[user])
        p1 = ET.SubElement(product, 'price')
        p1.text = str(price[user])
        desc1 = ET.SubElement(product, 'product_description')
        desc1.text = str(description,[user])
    tree = ET.ElementTree(root)
    
    tree.write(filepath + "Output.xml", encoding='utf-8', xml_declaration=True)    #write the tree into an XML file

In [18]:
# run code
if __name__ == '__main':
    url = input('Enter Website URL: ')
    file_path = input('Enter path to save scarping file: ')
    msg = input('Do you want to set number of pages? (press Y for yes or N for no) :')
    if msg == 'N':
        scrape_web(url)
        create_xml(file_path)
    elif msg == 'Y':
        n_page = input('Enter Number of pages :')
        scrape_web(url, int(n_page))
        create_xml(file_path)
    else:
        print('Invalid Value')
        
    print(url_web)